In [ ]:
# Starting of phase 1 that is Data Cleaning - preparing the dataset for analysis

In [ ]:
# Phase 1 - Step 2: Load the Dataset
import pandas as pd
import numpy as np

df = pd.read_csv('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')

# First look at the data
print("Shape of dataset:", df.shape)
print("\nFirst 5 rows:")
df.head()


In [ ]:
# Phase 1 - Step 3: Check Columns and Missing Values
print("Column names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values per column:")
print(df.isnull().sum())

In [ ]:
# Phase 1 - Step 4: Data Cleaning

# 1. Handle missing values in TotalCharges
print("Before cleaning - TotalCharges missing/blanks:", df['TotalCharges'].eq(' ').sum())

# Replace blank spaces with NaN
df['TotalCharges'] = df['TotalCharges'].replace(' ', np.nan)

# Convert to numeric
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

print("After cleaning - Missing values in TotalCharges:", df['TotalCharges'].isnull().sum())

# Fill missing TotalCharges with median (by contract type - more accurate)
df['TotalCharges'] = df.groupby('Contract')['TotalCharges'].transform(lambda x: x.fillna(x.median()))

print("\n✅ TotalCharges cleaned and converted to numeric!")
print(df['TotalCharges'].describe())

In [ ]:
# Phase 1 - Step 5: More Cleaning & Feature Engineering

# 1. Convert Yes/No columns to 1/0 (binary encoding)
binary_cols = ['Partner', 'Dependents', 'PhoneService', 'MultipleLines', 
               'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 
               'TechSupport', 'StreamingTV', 'StreamingMovies', 
               'PaperlessBilling', 'Churn']

for col in binary_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0, 'No internet service': 0, 'No phone service': 0})

print("✅ Binary columns encoded (Yes/No → 1/0)")

# 2. Create new useful features (as mentioned in roadmap)
df['AvgMonthlyCharge'] = df['TotalCharges'] / df['tenure'].replace(0, 1)  # avoid division by zero
df['TenureBand'] = pd.cut(df['tenure'], bins=[0, 12, 24, 48, 72, 100], 
                          labels=['0-12', '13-24', '25-48', '49-72', '73+'])

print("\n✅ New features created:")
print(df[['tenure', 'TotalCharges', 'AvgMonthlyCharge', 'TenureBand']].head())

# 3. Check final missing values and data types
print("\nRemaining missing values:")
print(df.isnull().sum().sum())  # should be 0 now

# 4. Save the cleaned dataset
df.to_csv('../data/telco_cleaned.csv', index=False)
print("\n✅ Cleaned dataset saved as 'telco_cleaned.csv' in data folder!")

In [ ]:
# Phase 1 - Step 6: Fix Remaining Issues & Final Cleaning

# Better binary encoding that handles all variations
binary_cols = ['Partner', 'Dependents', 'PhoneService', 'MultipleLines', 
               'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 
               'TechSupport', 'StreamingTV', 'StreamingMovies', 
               'PaperlessBilling', 'Churn']

for col in binary_cols:
    df[col] = df[col].replace({
        'Yes': 1, 
        'No': 0, 
        'No internet service': 0, 
        'No phone service': 0,
        ' ': 0
    }).astype(int)

# Re-fill any remaining TotalCharges (in case something slipped)
df['TotalCharges'] = df.groupby('Contract')['TotalCharges'].transform(
    lambda x: x.fillna(x.median())
)

print("✅ Improved binary encoding completed")

# Final check
print("\nRemaining missing values:", df.isnull().sum().sum())
print("\nData types summary:")
print(df.dtypes.value_counts())

# Save again
df.to_csv('../data/telco_cleaned.csv', index=False)
print("\n✅ Final cleaned dataset saved!")

In [ ]:
# Phase 1 - Step 7: Final Diagnosis & Complete Cleaning

# Find which columns still have missing values
print("Columns with missing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])

# Show a few rows with missing values
print("\nSample rows with missing data:")
print(df[df.isnull().any(axis=1)].head())

# Final comprehensive cleaning
# 1. Fix any remaining TotalCharges
df['TotalCharges'] = df.groupby('Contract')['TotalCharges'].transform(
    lambda x: x.fillna(x.median())
)

# 2. Fix all remaining object columns
object_cols = df.select_dtypes(include=['object']).columns
for col in object_cols:
    df[col] = df[col].replace(' ', np.nan).fillna('Unknown')

# 3. Re-encode binary columns more robustly
binary_cols = ['Partner', 'Dependents', 'PhoneService', 'MultipleLines', 
               'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 
               'TechSupport', 'StreamingTV', 'StreamingMovies', 
               'PaperlessBilling', 'Churn']

for col in binary_cols:
    if col in df.columns:
        df[col] = df[col].map({
            'Yes': 1, 
            'No': 0,
            'No internet service': 0,
            'No phone service': 0,
            'Unknown': 0
        }).fillna(0).astype(int)

print("\n✅ Final cleaning completed!")
print("Remaining missing values:", df.isnull().sum().sum())

# Save final version
df.to_csv('../data/telco_cleaned.csv', index=False)
print("✅ Final cleaned dataset saved successfully!")

In [ ]:
# Phase 1 - Step 8: Fix TenureBand and Complete Data Cleaning

# Fix TenureBand missing values
print("Before fix - Missing in TenureBand:", df['TenureBand'].isnull().sum())

# Fill missing TenureBand
df['TenureBand'] = df['TenureBand'].fillna('0-12')

# Final verification
print("After fix - Missing in TenureBand:", df['TenureBand'].isnull().sum())
print("\nTotal remaining missing values in entire dataset:", df.isnull().sum().sum())

# Final data quality check
print("\n✅ Final Data Quality Summary:")
print("Total rows:", len(df))
print("Total columns:", len(df.columns))
print("Missing values:", df.isnull().sum().sum())
print("\nData types:")
print(df.dtypes.value_counts())

# Save the final cleaned version
df.to_csv('../data/telco_cleaned.csv', index=False)
print("\n🎉 Phase 1 Completed! Cleaned dataset is ready.")

In [ ]:
# Phase 1 - Final Step: Drop Unnecessary Columns

print("Original shape:", df.shape)

# Drop columns we don't need
columns_to_drop = ['customerID']

df_clean = df.drop(columns=columns_to_drop, errors='ignore')

print("After dropping unnecessary columns:")
print("New shape:", df_clean.shape)
print("Remaining columns:", df_clean.columns.tolist())

# Optional: Drop any duplicate rows (if any)
df_clean = df_clean.drop_duplicates()
print("After removing duplicates - shape:", df_clean.shape)

# Save the final slimmed version
df_clean.to_csv('../data/telco_cleaned_final.csv', index=False)
print("\n✅ Slimmed dataset saved as 'telco_cleaned_final.csv'")

# Update df to use the cleaned version going forward
df = df_clean.copy()

In [16]:
print(df.isnull().sum()[df.isnull().sum() > 0])

TotalCharges    11
dtype: int64


In [17]:
df[df.isnull().any(axis=1)]

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,...,Yes,Yes,Yes,No,Two year,Yes,Bank transfer (automatic),52.55,NaN,0
753,3115-CZMZD,Male,0,No,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.25,NaN,0
936,5709-LVOEQ,Female,0,Yes,Yes,0,Yes,No,DSL,Yes,...,Yes,No,Yes,Yes,Two year,No,Mailed check,80.85,NaN,0
1082,4367-NUYAO,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.75,NaN,0
1340,1371-DWPAZ,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,...,Yes,Yes,Yes,No,Two year,No,Credit card (automatic),56.05,NaN,0
3331,7644-OMVMY,Male,0,Yes,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,19.85,NaN,0
3826,3213-VVOLG,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.35,NaN,0
4380,2520-SGTTA,Female,0,Yes,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.00,NaN,0
5218,2923-ARZLG,Male,0,Yes,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,One year,Yes,Mailed check,19.70,NaN,0
6670,4075-WKNIU,Female,0,Yes,Yes,0,Yes,Yes,DSL,No,...,Yes,Yes,Yes,No,Two year,No,Mailed check,73.35,NaN,0


In [18]:
# Fill missing TotalCharges with median
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())

# Verify again
print("Remaining missing values:", df.isnull().sum().sum())

Remaining missing values: 0
